In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii
Task: 08_retro_scoring


### Output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [4]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Get data

In [5]:
%%time

str_filename = 'df_pd_pre_with_yhats.gzip'
str_uri = f's3://{str_project}/ad_hoc/get_predictions/{str_variant}/{str_filename}'
df = pd.read_parquet(str_uri)

# subset
df = df[df['data_set'] == 'test'].copy()

# sort
df.sort_values(by=['bigaccountid__app','bitdebtor__app'], ascending=[True, False], inplace=True)

# show
df

CPU times: user 7.94 s, sys: 3.17 s, total: 11.1 s
Wall time: 3.51 s


,bigaccountid__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf105__tu,linkf195__tu,linkf193__tu,linkb012__tu,linkf032__tu,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_ad,yhat_pricing_pd,yhat_pricing_lgd
12401,3932700.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.09,1.282051,1.0,5.879452,0.258216,0.167898,0.619272
26946,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.06,1.322581,3.0,6.441096,0.490735,0.155154,0.638730
26947,3932720.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.06,1.322581,3.0,6.441096,0.479718,0.148102,0.638730
16514,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.15,1.195652,1.0,6.736986,0.266475,0.173194,0.607942
16515,3932724.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.165459,10,4,0.15,1.195652,1.0,6.736986,0.259319,0.163903,0.607942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7863,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.144717,12,4,0.09,1.548387,2.0,10.010959,0.336633,0.157515,0.592475
7864,4812498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,nan,...,1.144717,12,4,0.09,1.548387,2.0,10.010959,0.332483,0.146947,0.592475
20360,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,1.144717,12,4,0.12,1.044444,-1.0,0.208219,0.333434,0.172118,0.689688
20361,4812503.0,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,n,...,1.144717,12,4,0.12,1.044444,-1.0,0.208219,0.327937,0.146919,0.689688


### Get grouped scores at the account level

In [6]:
df_tmp = df.groupby(by='bigaccountid__app', as_index=False).agg({
    'intopenbktype__app': 'first',
    'yhat_pricing_pd': 'mean',
    'yhat_pricing_lgd': 'mean',
})

# ecnl
str_colname = 'ECNL_test'
if str_variant == 'noPTImodel10':
    df_tmp[str_colname] = (df_tmp['yhat_pricing_pd'] * df_tmp['yhat_pricing_lgd']) * 2.36
else:
    df_tmp[str_colname] = (df_tmp['yhat_pricing_pd'] * df_tmp['yhat_pricing_lgd'])

# show
df_tmp

,bigaccountid__app,intopenbktype__app,yhat_pricing_pd,yhat_pricing_lgd,ECNL_test
0,3932700.0,nan,0.167898,0.619272,0.245380
1,3932720.0,nan,0.151628,0.638730,0.228564
2,3932724.0,nan,0.168548,0.607942,0.241824
3,3932730.0,nan,0.578571,0.620471,0.847208
4,3932752.0,nan,0.366915,0.673531,0.583223
...,...,...,...,...,...
27802,4812483.0,7.0,0.103705,0.579950,0.141940
27803,4812497.0,nan,0.339801,0.597389,0.479065
27804,4812498.0,nan,0.152231,0.592475,0.212856
27805,4812503.0,7.0,0.159518,0.689688,0.259642


### Write aggregated data to s3

In [7]:
%%time

str_filename = 'df.csv'
str_uri = f's3://{str_project}/08_retro_scoring/08_segment_distributions/01_gen_xii_test/{str_filename}'
df_tmp.to_csv(str_uri, index=False)

CPU times: user 252 ms, sys: 237 ms, total: 489 ms
Wall time: 446 ms
